# Tier 2 Assignment 3 — Solutions: Hi-C Analysis and Motif Discovery

Complete solutions for all 8 problems.

In [ ]:
import numpy as np
import math
from math import log2, comb, sqrt

## Problem 1: Contact Matrix Normalization

Divide each entry by the geometric mean of its row and column sum.
Guard against zero row/column sums with an explicit check.

In [ ]:
def normalize_contact_matrix(matrix: list[list[int]]) -> list[list[float]]:
    n = len(matrix)
    row_sums = [sum(matrix[i]) for i in range(n)]
    col_sums = [sum(matrix[i][j] for i in range(n)) for j in range(n)]

    result = []
    for i in range(n):
        row = []
        for j in range(n):
            denom = sqrt(row_sums[i] * col_sums[j])
            row.append(matrix[i][j] / denom if denom != 0 else 0.0)
        result.append(row)
    return result


# Test
raw = [
    [4, 2, 0],
    [2, 6, 2],
    [0, 2, 4],
]
normed = normalize_contact_matrix(raw)
print("Normalized matrix:")
for row in normed:
    print([round(x, 4) for x in row])

## Problem 2: Contact Decay (P(s) Curve)

Walk each diagonal offset `d` from 0 to `n-1`, collect entries `matrix[i][i+d]`,
compute their mean, and pair with the distance in bp.

In [ ]:
def contact_decay(matrix: list[list[float]], bin_size: int) -> list[tuple]:
    n = len(matrix)
    result = []
    for d in range(n):
        entries = [matrix[i][i + d] for i in range(n - d)]
        avg = sum(entries) / len(entries)
        result.append((d * bin_size, avg))
    # Already in order; sort is a no-op but matches the spec
    return sorted(result, key=lambda x: x[0])


# Test
test_matrix = [
    [10, 5, 2, 1, 0],
    [ 5, 8, 4, 2, 1],
    [ 2, 4, 9, 5, 2],
    [ 1, 2, 5, 7, 4],
    [ 0, 1, 2, 4, 6],
]
decay = contact_decay(test_matrix, bin_size=10_000)
print("Contact decay P(s):")
for dist, avg in decay:
    print(f"  d={dist:>8} bp: avg_contact = {avg:.4f}")

## Problem 3: Insulation Score

Extract the `w×w` upstream-downstream submatrix for each bin `i` (where both
`i-w >= 0` and `i+w <= n`) and compute its mean. Edge bins get 0.0.

In [ ]:
def insulation_score(matrix: list[list[float]], window: int) -> list[float]:
    n = len(matrix)
    scores = []
    for i in range(n):
        if i < window or i > n - window - 1:
            scores.append(0.0)
            continue
        # Submatrix: rows i-w..i-1, cols i..i+w-1
        total = 0.0
        count = 0
        for r in range(i - window, i):
            for c in range(i, i + window):
                total += matrix[r][c]
                count += 1
        scores.append(total / count if count else 0.0)
    return scores


# Test
tad_matrix = [
    [10,  8,  1,  1,  0],
    [ 8, 10,  1,  0,  1],
    [ 1,  1, 10,  8,  1],
    [ 1,  0,  8, 10,  1],
    [ 0,  1,  1,  1, 10],
]
scores = insulation_score(tad_matrix, window=1)
print("Insulation scores:", [round(s, 4) for s in scores])
# Bins 1 and 2 are interior; bin at TAD boundary should be lowest
# insulation[1] = mean([[m[0][1]]) = 8; insulation[2] = mean([m[1][2]]) = 1
# -> bin 2 is the TAD boundary (score=1 is the minimum)

## Problem 4: A/B Compartment from Correlation

Convert the contact matrix to a numpy array, compute the Pearson correlation
matrix with `np.corrcoef`, center it, then use power iteration to find the
first eigenvector of `M @ M.T`.

In [ ]:
def ab_compartments(norm_matrix: list[list[float]]) -> list[int]:
    np.random.seed(42)
    A = np.array(norm_matrix, dtype=float)

    # Step 1: Pearson correlation matrix between rows
    C = np.corrcoef(A)

    # Step 2: Center
    M = C - C.mean(axis=0)

    # Step 3: Power iteration on M @ M.T
    cov = M @ M.T
    v = np.random.randn(cov.shape[0])
    v /= np.linalg.norm(v)
    for _ in range(50):
        v = cov @ v
        norm = np.linalg.norm(v)
        if norm == 0:
            break
        v /= norm

    # Step 4: Sign assignment
    return [1 if x > 0 else -1 for x in v]


# Test
ab_mat = [
    [10, 8, 7, 1, 1, 0],
    [ 8,10, 8, 1, 0, 1],
    [ 7, 8,10, 0, 1, 1],
    [ 1, 1, 0,10, 8, 7],
    [ 1, 0, 1, 8,10, 8],
    [ 0, 1, 1, 7, 8,10],
]
compartments = ab_compartments(ab_mat)
print("Compartment assignments:", compartments)
# Expect bins 0-2 all same sign, bins 3-5 opposite sign
assert len(set(compartments[:3])) == 1, "Bins 0-2 should share a compartment"
assert len(set(compartments[3:])) == 1, "Bins 3-5 should share a compartment"
assert compartments[0] != compartments[3], "The two blocks should differ"
print("Assertions passed.")

## Problem 5: PPM and PWM Construction

Count bases per position, add pseudocount 0.1 to each of the four bases,
divide by the adjusted total to get the PPM, then take log2 of the ratio
to the uniform background (0.25) to get the PWM.

In [ ]:
def build_ppm_pwm(sequences: list[str]) -> tuple[dict, dict]:
    bases = "ACGT"
    pseudocount = 0.1
    n_seqs = len(sequences)
    width = len(sequences[0])

    # Count occurrences
    counts = {b: [0] * width for b in bases}
    for seq in sequences:
        for pos, base in enumerate(seq.upper()):
            if base in counts:
                counts[base][pos] += 1

    # PPM with pseudocounts
    total = n_seqs + 4 * pseudocount
    ppm = {b: [(counts[b][p] + pseudocount) / total for p in range(width)]
           for b in bases}

    # PWM: log2(ppm / 0.25)
    pwm = {b: [log2(ppm[b][p] / 0.25) for p in range(width)]
           for b in bases}

    return ppm, pwm


# Test
motif_seqs = [
    "ACGT",
    "ACGT",
    "ACGT",
    "ACGT",
    "ACGA",
]
ppm, pwm = build_ppm_pwm(motif_seqs)
print("PPM:")
for base in "ACGT":
    print(f"  {base}: {[round(v, 4) for v in ppm[base]]}")
print("PWM:")
for base in "ACGT":
    print(f"  {base}: {[round(v, 4) for v in pwm[base]]}")

## Problem 6: Information Content

Apply the Shannon entropy formula at each position. Skip zero-probability
terms to avoid `log2(0)`. The `2 +` accounts for the maximum entropy of 2
bits for a 4-letter alphabet at uniform frequency.

In [ ]:
def information_content(ppm: dict) -> tuple[list[float], float]:
    bases = "ACGT"
    width = len(ppm["A"])
    ic_per_pos = []
    for p in range(width):
        entropy_sum = sum(
            ppm[b][p] * log2(ppm[b][p])
            for b in bases
            if ppm[b][p] > 0
        )
        ic_per_pos.append(2.0 + entropy_sum)
    total_ic = sum(ic_per_pos)
    return ic_per_pos, total_ic


# Test using PPM from Problem 5
ic_per_pos, total_ic = information_content(ppm)
print("IC per position:", [round(v, 4) for v in ic_per_pos])
print(f"Total IC: {total_ic:.4f} bits")

## Problem 7: PWM Sequence Scoring

Score every forward window by summing PWM columns. Build the reverse complement
and score it the same way, converting RC offsets back to original-strand positions
using `len(seq) - L - k`.

In [ ]:
_COMPLEMENT = str.maketrans("ACGTacgt", "TGCAtgca")


def _reverse_complement(seq: str) -> str:
    return seq.translate(_COMPLEMENT)[::-1]


def _score_window(pwm: dict, seq: str, start: int, motif_len: int) -> float:
    return sum(pwm[seq[start + j]][j] for j in range(motif_len))


def score_sequence(pwm: dict, sequence: str) -> dict:
    motif_len = len(pwm["A"])
    seq_len = len(sequence)
    best_score = float("-inf")
    best_position = 0
    best_strand = "+"

    # Forward strand
    for i in range(seq_len - motif_len + 1):
        # Skip windows containing bases not in the PWM
        window = sequence[i:i + motif_len].upper()
        if any(b not in pwm for b in window):
            continue
        s = sum(pwm[window[j]][j] for j in range(motif_len))
        if s > best_score:
            best_score = s
            best_position = i
            best_strand = "+"

    # Reverse complement strand
    rc = _reverse_complement(sequence).upper()
    for k in range(seq_len - motif_len + 1):
        window = rc[k:k + motif_len]
        if any(b not in pwm for b in window):
            continue
        s = sum(pwm[window[j]][j] for j in range(motif_len))
        if s > best_score:
            best_score = s
            best_position = seq_len - motif_len - k
            best_strand = "-"

    return {"best_score": best_score, "best_position": best_position, "strand": best_strand}


# Test
pwm_seqs = ["ACGT"] * 10
_, test_pwm = build_ppm_pwm(pwm_seqs)

test_seq = "NNNACGTNNN"
result = score_sequence(test_pwm, test_seq)
print("Forward motif hit:", result)
assert result["best_position"] == 3, f"Expected position 3, got {result['best_position']}"
assert result["strand"] == "+", f"Expected strand +, got {result['strand']}"

# Reverse complement test: ACGT is a palindrome, so RC hit also scores perfectly
rc_seq = "NNNACGTNNN"
rc_result = score_sequence(test_pwm, rc_seq)
print("RC test:", rc_result)
print("All assertions passed.")

## Problem 8: Fisher's Exact Motif Enrichment

Use `score_sequence` to label each sequence as a hit or miss, build the 2×2
contingency table, compute the odds ratio, then sum hypergeometric probabilities
for all tables as extreme or more extreme using `math.comb`.

The hypergeometric formula:
```
P(k) = C(fg_total, k) * C(bg_total, total_hits - k) / C(n, total_hits)
```
Sum for `k = fg_hits .. min(fg_total, total_hits)`.

In [ ]:
def motif_enrichment(
    foreground: list[str],
    background: list[str],
    pwm: dict,
    threshold: float,
) -> dict:
    def is_hit(seq: str) -> bool:
        res = score_sequence(pwm, seq)
        return res["best_score"] >= threshold

    a = sum(1 for s in foreground if is_hit(s))   # fg hits
    b = len(foreground) - a                        # fg misses
    c = sum(1 for s in background if is_hit(s))   # bg hits
    d = len(background) - c                        # bg misses
    n = a + b + c + d
    total_hits = a + c

    # Odds ratio
    denom = b * c
    odds_ratio = (a * d) / denom if denom != 0 else 0.0

    # Fisher's exact p-value (one-tailed: enrichment in fg)
    # P(k) = C(fg_total, k) * C(bg_total, total_hits - k) / C(n, total_hits)
    fg_total = a + b
    bg_total = c + d
    denominator = comb(n, total_hits)

    p_value = 0.0
    if denominator > 0:
        for k in range(a, min(fg_total, total_hits) + 1):
            bg_k = total_hits - k
            if bg_k < 0 or bg_k > bg_total:
                continue
            p_value += comb(fg_total, k) * comb(bg_total, bg_k) / denominator

    return {
        "odds_ratio": round(odds_ratio, 6),
        "p_value": round(p_value, 6),
        "fg_hits": a,
        "bg_hits": c,
    }


# Test
fg_seqs = [
    "TTTACGTAAA",
    "AAACGTTTT",
    "GGACGTCC",
    "TTACGTTT",
    "CCACGTGG",
    "AAATTTGGG",
]
bg_seqs = [
    "AAATTTGGG",
    "CCCGGGTTT",
    "TTTGGGCCC",
    "GGGAAACCC",
    "TTTACGTAA",
    "AAACCCGGG",
    "CCCAAATTT",
    "TTTCCCAAA",
]

tight_pwm_seqs = ["ACGT"] * 20
_, tight_pwm = build_ppm_pwm(tight_pwm_seqs)

result = motif_enrichment(fg_seqs, bg_seqs, tight_pwm, threshold=3.0)
print("Enrichment result:", result)
assert result["fg_hits"] > 0, "Should have foreground hits"
assert result["p_value"] <= 1.0, "p-value must be in [0, 1]"
print("Assertions passed.")

---

## Key Patterns

1. **Contact normalization**: geometric mean of marginals is the simplest coverage bias correction; zero-sum rows/columns are dead bins and should yield 0.
2. **P(s) curve**: iterate over diagonal offsets `d`; the number of entries at offset `d` is `n - d`.
3. **Insulation score**: the `[i-w:i, i:i+w]` block is strictly upstream rows × downstream columns — get the indices right before summing.
4. **Power iteration**: converges to the leading eigenvector after ~50 steps for well-conditioned matrices; always re-normalize after each multiply.
5. **Pseudocounts**: add a small constant to every base count before dividing to avoid log(0) in the PWM.
6. **Reverse complement**: translate then reverse; position on the original strand = `len(seq) - L - k`.
7. **Fisher's exact test**: sum hypergeometric probabilities starting at the observed count, not from 0; use `math.comb` to avoid floating-point issues with factorials.